# MP4 to Animated WebP Converter

Upload an MP4 file and convert it to animated WebP with easy trimming, cropping, and quality controls.

In [ ]:
# Install dependencies
!pip install -q opencv-python-headless pillow

import cv2
import numpy as np
from PIL import Image
from google.colab import files
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
import io
import base64
import os

In [ ]:
# Upload MP4 file
print("Please upload your MP4 file:")
uploaded = files.upload()
video_filename = list(uploaded.keys())[0]
print(f"Uploaded: {video_filename}")

In [ ]:
# Get video info
cap = cv2.VideoCapture(video_filename)
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
fps = cap.get(cv2.CAP_PROP_FPS)
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
duration = total_frames / fps
cap.release()

print(f"Video Info:")
print(f"  Resolution: {width} x {height}")
print(f"  Total Frames: {total_frames}")
print(f"  FPS: {fps:.2f}")
print(f"  Duration: {duration:.2f} seconds")

In [ ]:
# Create UI Controls
style = {'description_width': '120px'}
layout = widgets.Layout(width='500px')

# Frame trimming
frame_range = widgets.IntRangeSlider(
    value=[0, total_frames - 1],
    min=0,
    max=total_frames - 1,
    step=1,
    description='Frame Range:',
    style=style,
    layout=layout,
    continuous_update=False
)

# Crop controls
crop_x = widgets.IntRangeSlider(
    value=[0, width],
    min=0,
    max=width,
    step=1,
    description='Crop X (L-R):',
    style=style,
    layout=layout,
    continuous_update=False
)

crop_y = widgets.IntRangeSlider(
    value=[0, height],
    min=0,
    max=height,
    step=1,
    description='Crop Y (T-B):',
    style=style,
    layout=layout,
    continuous_update=False
)

# Quality setting
quality = widgets.IntSlider(
    value=70,
    min=1,
    max=100,
    step=1,
    description='Quality (%):',
    style=style,
    layout=layout
)

# Loop option
loop = widgets.Checkbox(
    value=True,
    description='Loop Animation',
    style=style
)

# Output scale
scale = widgets.FloatSlider(
    value=1.0,
    min=0.1,
    max=2.0,
    step=0.1,
    description='Output Scale:',
    style=style,
    layout=layout
)

# Frame skip (for smaller file size)
frame_skip = widgets.IntSlider(
    value=1,
    min=1,
    max=10,
    step=1,
    description='Frame Skip:',
    style=style,
    layout=layout
)

# Preview output
preview_output = widgets.Output()

# Preview button
preview_btn = widgets.Button(description='Preview Frame', button_style='info')

def show_preview(b):
    with preview_output:
        clear_output(wait=True)
        cap = cv2.VideoCapture(video_filename)
        start_frame = frame_range.value[0]
        cap.set(cv2.CAP_PROP_POS_FRAMES, start_frame)
        ret, frame = cap.read()
        cap.release()
        
        if ret:
            # Apply crop
            x1, x2 = crop_x.value
            y1, y2 = crop_y.value
            frame = frame[y1:y2, x1:x2]
            
            # Apply scale
            if scale.value != 1.0:
                new_w = int(frame.shape[1] * scale.value)
                new_h = int(frame.shape[0] * scale.value)
                frame = cv2.resize(frame, (new_w, new_h))
            
            # Convert to RGB for display
            frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            img = Image.fromarray(frame_rgb)
            
            # Display preview
            buffer = io.BytesIO()
            img.save(buffer, format='PNG')
            img_str = base64.b64encode(buffer.getvalue()).decode()
            
            display(HTML(f'<img src="data:image/png;base64,{img_str}" style="max-width:600px"/>'))
            print(f"Preview: Frame {start_frame}, Size: {img.width}x{img.height}")

preview_btn.on_click(show_preview)

# Display UI
print("=" * 50)
print("CONVERSION SETTINGS")
print("=" * 50)
display(widgets.VBox([
    widgets.HTML('<h4>Trim Frames</h4>'),
    frame_range,
    widgets.HTML('<h4>Crop Region</h4>'),
    crop_x,
    crop_y,
    widgets.HTML('<h4>Output Settings</h4>'),
    quality,
    scale,
    frame_skip,
    loop,
    widgets.HBox([preview_btn]),
    preview_output
]))

In [ ]:
# Convert to WebP
print("Converting to animated WebP...")
print(f"Settings:")
print(f"  Frames: {frame_range.value[0]} to {frame_range.value[1]}")
print(f"  Crop X: {crop_x.value[0]} to {crop_x.value[1]}")
print(f"  Crop Y: {crop_y.value[0]} to {crop_y.value[1]}")
print(f"  Quality: {quality.value}%")
print(f"  Scale: {scale.value}x")
print(f"  Frame Skip: {frame_skip.value}")
print(f"  Loop: {loop.value}")
print()

# Extract frames
cap = cv2.VideoCapture(video_filename)
frames = []

start_frame, end_frame = frame_range.value
x1, x2 = crop_x.value
y1, y2 = crop_y.value

cap.set(cv2.CAP_PROP_POS_FRAMES, start_frame)

frame_count = 0
total_to_process = end_frame - start_frame + 1

for i in range(start_frame, end_frame + 1):
    ret, frame = cap.read()
    if not ret:
        break
    
    # Skip frames if needed
    if (i - start_frame) % frame_skip.value != 0:
        continue
    
    # Crop
    frame = frame[y1:y2, x1:x2]
    
    # Scale
    if scale.value != 1.0:
        new_w = int(frame.shape[1] * scale.value)
        new_h = int(frame.shape[0] * scale.value)
        frame = cv2.resize(frame, (new_w, new_h))
    
    # Convert BGR to RGB
    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    frames.append(Image.fromarray(frame_rgb))
    
    frame_count += 1
    if frame_count % 50 == 0:
        print(f"Processed {frame_count} frames...")

cap.release()
print(f"Total frames extracted: {len(frames)}")

# Calculate duration per frame in milliseconds
effective_fps = fps / frame_skip.value
duration_ms = int(1000 / effective_fps)

# Save as animated WebP
output_filename = os.path.splitext(video_filename)[0] + '.webp'

frames[0].save(
    output_filename,
    'WEBP',
    save_all=True,
    append_images=frames[1:],
    duration=duration_ms,
    loop=0 if loop.value else 1,
    quality=quality.value
)

file_size = os.path.getsize(output_filename) / (1024 * 1024)
print(f"\nSaved: {output_filename}")
print(f"File size: {file_size:.2f} MB")

In [ ]:
# Preview the result
from IPython.display import Image as IPImage, display

print("Preview of generated WebP:")
display(IPImage(filename=output_filename))

In [ ]:
# Download the result
print("Downloading the WebP file...")
files.download(output_filename)